In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 5 - WEEK 10 BAYESIAN OPTIMISATION
# Run from inside the week10/ folder
# ============================================================
#
# Strategy:
# - Manual Y standardisation.
# - Refit ARD Matern GP including Week 9.
# - Check Week 9 calibration.
# - Concentrate search near the proven upper-corner basin.
# - Explicitly sample the x1=x2=x3=1 boundary line in x4.
# - Retain a global pool only as a diagnostic.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 10 cumulative data
# ------------------------------------------------------------

X = np.load("function5/initial_inputs.npy")
Y = np.load("function5/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. WEEK 9 CALIBRATION CHECK
# ------------------------------------------------------------
#
# Week 9 selected:
# [1.000000, 1.000000, 1.000000, 0.979991]
#
# Week 9 GP prediction:
# mean ≈ 8460.837
# std  ≈ 137.982
#
# Actual:
# 8290.218689357127
# ------------------------------------------------------------

week9_pred_mean = 8460.837479980311
week9_pred_std = 137.98176799506345
week9_actual = 8290.218689357127

week9_error = (
    week9_actual
    - week9_pred_mean
)

week9_z_error = (
    week9_error
    / week9_pred_std
)

print("\n================================")
print("WEEK 9 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week9_pred_mean)
print("Predicted std :", week9_pred_std)
print("Actual        :", week9_actual)

print("\nPrediction error:")
print(week9_error)

print("\nError / predicted std:")
print(week9_z_error)


# ------------------------------------------------------------
# 3. Manual Y standardisation
# ------------------------------------------------------------

y_mean = np.mean(Y)
y_std = np.std(Y)

Y_scaled = (
    Y - y_mean
) / y_std

best_y_scaled = np.max(Y_scaled)

print("\n================================")
print("Y STANDARDISATION")
print("================================")

print("Y mean:", y_mean)
print("Y std :", y_std)

print("\nBest scaled Y:")
print(best_y_scaled)


# ------------------------------------------------------------
# 4. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y_scaled
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.20 * lengthscales,
    0.015,
    0.08
)

wide_scale = np.clip(
    0.40 * lengthscales,
    0.04,
    0.15
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


# Local around actual incumbent [1,1,1,1]

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(120000, 4)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(80000, 4)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(80000, 4)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)


# ------------------------------------------------------------
# 7. Explicit boundary-line coverage
# ------------------------------------------------------------
#
# Proven strong region:
# x1 = x2 = x3 = 1
#
# Sweep x4 densely near the upper boundary.
# ------------------------------------------------------------

x4_line = np.linspace(
    0.80,
    1.0,
    10001
)

boundary_line = np.column_stack([
    np.ones_like(x4_line),
    np.ones_like(x4_line),
    np.ones_like(x4_line),
    x4_line
])


# Combine all pools

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates,
    boundary_line
])


# ------------------------------------------------------------
# 8. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 9. GP predictions
# ------------------------------------------------------------

mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

mu_raw = (
    mu_scaled * y_std
    + y_mean
)

sigma_raw = (
    sigma_scaled * y_std
)


# ------------------------------------------------------------
# 10. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("raw mean =", mu_raw[ei_idx])
print("raw std =", sigma_raw[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 11. EI sensitivity
# ------------------------------------------------------------

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in [
    0.0,
    0.01,
    0.05,
    0.10
]:

    EI_test = expected_improvement(
        mu_scaled,
        sigma_scaled,
        best_y_scaled,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", xi,
        "\n candidate =", candidates[idx],
        "\n raw mean =", round(mu_raw[idx], 3),
        "\n raw std =", round(sigma_raw[idx], 3),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 12. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu_scaled)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("raw mean =", mu_raw[mean_idx])
print("raw std =", sigma_raw[mean_idx])


# ------------------------------------------------------------
# 13. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n raw mean =", round(mu_raw[idx], 3),
        "\n raw std =", round(sigma_raw[idx], 3),
        "\n scaled UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 14. Distance from incumbent
# ------------------------------------------------------------

def distance_from_best(x):
    return np.linalg.norm(
        x - best_x
    )

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    distance_from_best(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    distance_from_best(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        distance_from_best(
            candidates[idx]
        )
    )


# ------------------------------------------------------------
# 15. Upper-boundary diagnostic
# ------------------------------------------------------------

def upper_boundary_count(
    x,
    tol=0.01
):

    return np.sum(
        x >= 1.0 - tol
    )


print("\n================================")
print("UPPER-BOUNDARY CHECK")
print("================================")

print(
    "EI dimensions near 1:",
    upper_boundary_count(
        candidates[ei_idx]
    )
)

print(
    "Highest mean dimensions near 1:",
    upper_boundary_count(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta} dimensions near 1:",
        upper_boundary_count(
            candidates[idx]
        )
    )

DATA
X shape: (29, 4)
Y shape: (29,)

Current best:
[1. 1. 1. 1.] -> 8662.4825

Y range:
min = 0.1129397953712203
max = 8662.4825
std = 2943.6526283605567

WEEK 9 CALIBRATION CHECK
Predicted mean: 8460.837479980311
Predicted std : 137.98176799506345
Actual        : 8290.218689357127

Prediction error:
-170.6187906231844

Error / predicted std:
-1.236531413550873

Y STANDARDISATION
Y mean: 1899.8421085175569
Y std : 2943.6526283605567

Best scaled Y:
2.2973635972967505


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 18 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.68**2 * Matern(length_scale=[1.19, 1.24, 1.21, 2], nu=2.5) + WhiteKernel(noise_level=0.0022)

ARD lengthscales:
[1.1931252  1.23682301 1.20586087 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.28163695 0.27168652 0.27866245 0.16801407]

CANDIDATE SCALES
Local widths:
[0.08 0.08 0.08 0.08]

Wide widths:
[0.15 0.15 0.15 0.15]

Candidates after duplicate filtering:
266613

PRIMARY EI
candidate = [1.         1.         1.         0.36751931]
raw mean = 6050.281294562121
raw std = 1105.0165607313213
EI = 0.0011377932912141325

EI SENSITIVITY

xi = 0.0 
 candidate = [1.         1.         1.         0.36751931] 
 raw mean = 6050.281 
 raw std = 1105.017 
 EI = 0.00113779 

xi = 0.01 
 candidate = [1.         1.         1.         0.36751931] 
 raw mean = 6050.281 
 raw std = 1105.017 
 EI = 0.00105057 

xi = 0.05 
 candidate = [1.         1.         1.         0.36751931] 
 raw mean = 6050.281 
 raw std = 1105.017 
 EI = 0.00075902 

xi = 0.1 
 candidat

In [2]:
# ============================================================
# FINAL FUNCTION 5 - WEEK 10 SELECTION
# ============================================================
#
# Week 9 calibration error was approximately -1.24 sigma:
# mildly optimistic, but not a major surrogate failure.
#
# Global EI is rejected because it moves far away from the
# incumbent and is dominated by uncertainty.
#
# Highest posterior mean and ALL tested UCB beta values select
# the same point immediately outside the 0.01 duplicate radius
# around the actual best [1,1,1,1].
#
# Interpretation:
# the model continues to favour the upper corner; x3=0.98997
# should not be interpreted as a newly identified optimum.
# It is the best admissible nearby point under our duplicate
# filtering rule.

final_idx = np.argmax(mu_scaled)

week10_candidate = candidates[final_idx]

print("Week 10 Function 5 candidate:")
print(week10_candidate)

print("\nPredicted raw mean:")
print(mu_raw[final_idx])

print("\nPredicted raw std:")
print(sigma_raw[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week10_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week10_candidate
)

print("\nPortal format:")
print(portal)

Week 10 Function 5 candidate:
[1.         1.         0.98997095 1.        ]

Predicted raw mean:
8390.704477928135

Predicted raw std:
162.1022663093704

Distance from current best:
0.010029051801230371

Portal format:
1.000000-1.000000-0.989971-1.000000
